In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [ ]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [ ]:
df

In [ ]:
df.sample(5)

In [ ]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [ ]:
train_y.value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_features=2000)

train_x_vector = tfidf.fit_transform(train_x)

test_x_vector = tfidf.transform(test_x)


In [ ]:
train_x.shape

In [ ]:
train_x_vector.shape

In [ ]:
type(train_x_vector)

In [ ]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [ ]:
primera_resenia

In [ ]:
train_x.iloc[0]

In [ ]:
primera_resenia[primera_resenia != 0]

In [ ]:
train_x.iloc[0]

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

In [ ]:
print(svc.score(test_x_vector, test_y))

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'SVC (linear)': SVC(kernel='linear'),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gaussian NB': GaussianNB()
}

results = {}

# GaussianNB requiere matrices densas.
dense_train = train_x_vector.toarray()
dense_test = test_x_vector.toarray()

for name, model in models.items():
    if name == 'Gaussian NB':
        model.fit(dense_train, train_y)
        y_pred = model.predict(dense_test)
    else:
        model.fit(train_x_vector, train_y)
        y_pred = model.predict(test_x_vector)

    results[name] = {
        'accuracy': accuracy_score(test_y, y_pred),
        'report': classification_report(test_y, y_pred, labels=['positive', 'negative']),
        'confusion': confusion_matrix(test_y, y_pred, labels=['positive', 'negative'])
    }

for name, res in results.items():
    print('=' * 30)
    print(name)
    print('Accuracy:', res['accuracy'])
    print(res['report'])
    print('Confusion matrix:')
    print(res['confusion'])
    print()
